<a href="https://colab.research.google.com/github/LuizFellipiFreire25/Projeto-ECAA08/blob/main/08%20-%20Base%20de%20Conhecimento%20e%20Regras%20de%20Diagnostico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 08 - Notebook: Sistemas Especialistas — Base de Conhecimento e Regras de Diagnóstico do AGV

Este notebook implementa a arquitetura de um **Sistema Especialista Baseado em Regras (RBS)** para o SCADA-Core do AGV. Ele modela a Base de Fatos dinâmicos, a Base de Conhecimento estruturada em Cláusulas de Horn, o Motor de Inferência por Encademaneto para Frente (*Forward Chaining*), arbitragem de conflitos por prioridade IEC 61508 e validações de consistência lógica.

---

### Célula 1 (Texto / Markdown)
```markdown
## 1. Definição das Estruturas de Dados do Sistema Especialista

Implementação das classes `FatoTelemetria` (representando proposições capturadas do robô) e `RegraSegurancaAGV` (Cláusulas de Horn com metadados de prioridade e tempo de resposta mecatrônico).

In [1]:
from datetime import datetime
from typing import List, Dict, Set, Optional
import pandas as pd

class FatoTelemetria:
    """Representa um fato ou evento proposicional registrado pela telemetria do AGV."""
    def __init__(self, nome: str, valor: bool = True, fonte: str = "SCADA_CORE"):
        self.nome = nome
        self.valor = valor
        self.fonte = fonte
        self.timestamp = datetime.now().strftime("%H:%M:%S.%f")[:-3]

    def __repr__(self):
        return f"[{self.timestamp}] {self.nome} = {self.valor} ({self.fonte})"


class RegraSegurancaAGV:
    """Representa uma Cláusula de Horn Definida com parâmetros de segurança industrial."""
    def __init__(
        self,
        id_regra: str,
        antecedentes: List[str],
        consequente: str,
        causa_raiz: str,
        severidade: str,
        prioridade: int,
        tempo_resposta_ms: int
    ):
        self.id_regra = id_regra
        self.antecedentes = antecedentes  # Lista de proposições necessárias (AND)
        self.consequente = consequente    # Proposição inferida / comando gerado
        self.causa_raiz = causa_raiz
        self.severidade = severidade
        self.prioridade = prioridade      # Maior valor = Maior prioridade no desempate
        self.tempo_resposta_ms = tempo_resposta_ms

    def avaliar_antecedentes(self, fatos_ativos: Set[str]) -> bool:
        """Verifica se TODOS os antecedentes estão presentes na Base de Fatos atual."""
        return all(ant in fatos_ativos for ant in self.antecedentes)

    def __repr__(self):
        ants = " ∧ ".join(self.antecedentes)
        return f"[{self.id_regra}] SE ({ants}) ENTÃO {self.consequente} (Prio: {self.prioridade})"

print("Estruturas FatoTelemetria e RegraSegurancaAGV definidas com sucesso.")

Estruturas FatoTelemetria e RegraSegurancaAGV definidas com sucesso.


## 2. Motor de Inferência e Gerenciador da Base de Conhecimento

A classe `MotorInferenciaAGV` armazena as regras e os fatos dinâmicos. Ela executa a dedução por **Encadeamento para Frente (*Forward Chaining*)**, gerencia o Conjunto de Conflitos e prioriza ações críticas de segurança (SIL 3).

In [2]:
class MotorInferenciaAGV:
    """Motor de inferência baseado em regras com resolução de conflitos e validação de consistência."""
    def __init__(self):
        self.base_conhecimento: List[RegraSegurancaAGV] = []
        self.fatos_dinamicos: Dict[str, FatoTelemetria] = {}
        self.log_disparos: List[Dict] = []

    def adicionar_regra(self, regra: RegraSegurancaAGV):
        """Insere uma regra na Base de Conhecimento."""
        self.base_conhecimento.append(regra)

    def afirmat_fato(self, nome_fato: str, fonte: str = "SENSOR_CAMPO"):
        """Adiciona ou atualiza um fato na Base de Fatos do AGV."""
        fato = FatoTelemetria(nome=nome_fato, valor=True, fonte=fonte)
        self.fatos_dinamicos[nome_fato] = fato

    def reset_fatos(self):
        """Limpa a memória de trabalho (Base de Fatos) mantendo as regras."""
        self.fatos_dinamicos.clear()
        self.log_disparos.clear()

    def verificar_integridade_base() -> List[str]:
        """Detecta contradições e redundâncias potenciais na Base de Conhecimento."""
        alertas = []
        # Verifica regras idênticas com consequentes conflitantes
        for i, r1 in enumerate(self.base_conhecimento):
            for r2 in self.base_conhecimento[i+1:]:
                if set(r1.antecedentes) == set(r2.antecedentes):
                    if r1.consequente != r2.consequente:
                        alertas.append(f"CONTRADIÇÃO DETECTADA: {r1.id_regra} e {r2.id_regra} possuem mesmos antecedentes mas consequentes opostos!")
        return alertas

    def executar_forward_chaining(self) -> Dict:
        """Executa a máquina de inferência até atingir um ponto fixo (nenhuma regra nova disparada)."""
        fatos_ativos = set(self.fatos_dinamicos.keys())
        regras_disparadas_ids = set()
        loop_count = 0

        while True:
            loop_count += 1
            # 1. Pattern Matching: Encontra todas as regras cujos antecedentes são satisfeitos
            agenda = []
            for regra in self.base_conhecimento:
                if regra.id_regra not in regras_disparadas_ids:
                    if regra.avaliar_antecedentes(fatos_ativos):
                        agenda.append(regra)

            # Se nenhuma regra nova for ativada, o raciocínio encerrou
            if not agenda:
                break

            # 2. Resolução de Conflitos: Ordena por Prioridade (Decrescente) e Tempo de Resposta (Crescente)
            agenda.sort(key=lambda r: (-r.prioridade, r.tempo_resposta_ms))

            # 3. Disparo da Regra de Maior Prioridade (Vencedora da Arbitragem)
            regra_vencedora = agenda[0]
            regras_disparadas_ids.add(regra_vencedora.id_regra)

            # 4. Inferência de Novo Fato
            novo_fato_nome = regra_vencedora.consequente
            if novo_fato_nome not in fatos_ativos:
                self.afirmat_fato(novo_fato_nome, fonte=f"INFERENCIA_{regra_vencedora.id_regra}")
                fatos_ativos.add(novo_fato_nome)

            # Registro histórico
            self.log_disparos.append({
                "Ciclo": loop_count,
                "Regra Disparada": regra_vencedora.id_regra,
                "Antecedentes": " ∧ ".join(regra_vencedora.antecedentes),
                "Novo Fato Inferido": novo_fato_nome,
                "Causa-Raiz": regra_vencedora.causa_raiz,
                "Severidade": regra_vencedora.severidade,
                "Prioridade SIL": regra_vencedora.prioridade,
                "Tempo Resposta (ms)": regra_vencedora.tempo_resposta_ms
            })

        return {
            "fatos_finais": list(fatos_ativos),
            "historico": self.log_disparos
        }

print("MotorInferenciaAGV compilado e pronto para operação.")

MotorInferenciaAGV compilado e pronto para operação.


## 3. Carregamento da Base de Conhecimento Especialista (Catálogo R-01 a R-06)

Popula o motor com as regras de produção extraídas da análise mecatrônica de segurança do AGV.

In [3]:
motor = MotorInferenciaAGV()

# Cadastro do Catálogo de Regras Industriais (Tabela da Seção 2)
regras_catalogo = [
    RegraSegurancaAGV(
        id_regra="R-01",
        antecedentes=["LIDAR_ZONA_VERMELHA", "VELOCIDADE_GT_ZERO"],
        consequente="RISCO_COLISAO_IMINENTE",
        causa_raiz="Intrusão de Obstáculo Dinâmico na Rota",
        severidade="CRÍTICA (SIL 3)",
        prioridade=10,
        tempo_resposta_ms=50
    ),
    RegraSegurancaAGV(
        id_regra="R-02",
        antecedentes=["RISCO_COLISAO_IMINENTE", "FALHA_COMUNIC_WIFI"],
        consequente="TRIP_ISOLAMENTO_TOTAL",
        causa_raiz="AGV Cego e Incomunicável em Rota de Colisão",
        severidade="EMERGÊNCIA",
        prioridade=10,
        tempo_resposta_ms=10
    ),
    RegraSegurancaAGV(
        id_regra="R-03",
        antecedentes=["MOTOR_PWM_HIGH", "ENCODER_RPM_ZERO"],
        consequente="ROTOR_BLOQUEADO",
        causa_raiz="Travamento Mecânico da Roda de Tração",
        severidade="ALTA",
        prioridade=7,
        tempo_resposta_ms=100
    ),
    RegraSegurancaAGV(
        id_regra="R-04",
        antecedentes=["SENSOR_GAS_NH3_HIGH"],
        consequente="ABORTAR_MISSAO_QUIMICA",
        causa_raiz="Entrada em Zona de Vazamento Tóxico de Amônia",
        severidade="CRÍTICA",
        prioridade=9,
        tempo_resposta_ms=200
    ),
    RegraSegurancaAGV(
        id_regra="R-05",
        antecedentes=["BATERIA_SOC_LOW", "CARGA_ENGATADA"],
        consequente="BMS_DESCARGA_CRITICA",
        causa_raiz="Risco de Apagão com Carga Pesada no Trajeto",
        severidade="MÉDIA",
        prioridade=4,
        tempo_resposta_ms=500
    ),
    RegraSegurancaAGV(
        id_regra="R-06",
        antecedentes=["BMS_TEMP_HIGH", "CORRENTE_CARGA_HIGH"],
        consequente="RISCO_FUGA_TERMICA",
        causa_raiz="Sobreaquecimento Severo do Pack de Lítio",
        severidade="CRÍTICA",
        prioridade=8,
        tempo_resposta_ms=150
    )
]

for r in regras_catalogo:
    motor.adicionar_regra(r)

print(f"Base de Conhecimento carregada com {len(motor.base_conhecimento)} regras de segurança.")

Base de Conhecimento carregada com 6 regras de segurança.


## 4. Simulação de Emergência Cadenciada (Cadeia de Inferência Múltipla)

Simulação de uma situação de emergência onde o LiDAR detecta um obstáculo (`LIDAR_ZONA_VERMELHA`), o veículo está em movimento (`VELOCIDADE_GT_ZERO`) e simultaneamente perde o sinal de Wi-Fi (`FALHA_COMUNIC_WIFI`). O motor demonstrará o raciocínio em múltiplos níveis (*Forward Chaining*).

In [4]:
# Reset da memória do robô
motor.reset_fatos()

# Injeção de Fatos de Telemetria de Campo
print("--- [TELEMETRIA DE CAMPO RECEBIDA] ---")
motor.afirmat_fato("LIDAR_ZONA_VERMELHA", fonte="LiDAR_Safety_S300")
motor.afirmat_fato("VELOCIDADE_GT_ZERO", fonte="Encoder_Tração")
motor.afirmat_fato("FALHA_COMUNIC_WIFI", fonte="Watchdog_Modem")

# Exibe Fatos Iniciais
for f in motor.fatos_dinamicos.values():
    print(f)

print("\n--- [EXECUTANDO MOTOR DE INFERÊNCIA] ---")
resultado = motor.executar_forward_chaining()

print("\n--- [RELATÓRIO DE INFERÊNCIA / ARBITRAGEM] ---")
df_log = pd.DataFrame(resultado["historico"])
display(df_log)

print("\nBase de Fatos Final Resultante (Fatos de Telemetria + Fatos Inferidos):")
for fato_nome in resultado["fatos_finais"]:
    print(f" - {motor.fatos_dinamicos[fato_nome]}")

--- [TELEMETRIA DE CAMPO RECEBIDA] ---
[18:58:48.140] LIDAR_ZONA_VERMELHA = True (LiDAR_Safety_S300)
[18:58:48.140] VELOCIDADE_GT_ZERO = True (Encoder_Tração)
[18:58:48.141] FALHA_COMUNIC_WIFI = True (Watchdog_Modem)

--- [EXECUTANDO MOTOR DE INFERÊNCIA] ---

--- [RELATÓRIO DE INFERÊNCIA / ARBITRAGEM] ---


,Ciclo,Regra Disparada,Antecedentes,Novo Fato Inferido,Causa-Raiz,Severidade,Prioridade SIL,Tempo Resposta (ms)
0,1,R-01,LIDAR_ZONA_VERMELHA ∧ VELOCIDADE_GT_ZERO,RISCO_COLISAO_IMINENTE,Intrusão de Obstáculo Dinâmico na Rota,CRÍTICA (SIL 3),10,50
1,2,R-02,RISCO_COLISAO_IMINENTE ∧ FALHA_COMUNIC_WIFI,TRIP_ISOLAMENTO_TOTAL,AGV Cego e Incomunicável em Rota de Colisão,EMERGÊNCIA,10,10



Base de Fatos Final Resultante (Fatos de Telemetria + Fatos Inferidos):
 - [18:58:48.141] TRIP_ISOLAMENTO_TOTAL = True (INFERENCIA_R-02)
 - [18:58:48.141] RISCO_COLISAO_IMINENTE = True (INFERENCIA_R-01)
 - [18:58:48.140] VELOCIDADE_GT_ZERO = True (Encoder_Tração)
 - [18:58:48.140] LIDAR_ZONA_VERMELHA = True (LiDAR_Safety_S300)
 - [18:58:48.141] FALHA_COMUNIC_WIFI = True (Watchdog_Modem)
